# Auto ARIMA Benchmark — smooth 1.0.9 (2026-09-27)

Rerun of the auto-ARIMA comparison on M1 + M3 + Tourism (5,315 series) with the
Hannan-Rissanen starting values of the AR / MA parameters (`arima-initialisation`
branch), against the previous smooth run of 2026-09-24. Same harness as before:
point accuracy (**RMSSE**) and **computational time**. The predictive-distribution
cells are dropped here: the pinball arrays of the previous runs are not in the
repository, so there is nothing to compare them against.

**Only the smooth rows were recomputed.** The previous smooth rows are read from
`2026-09-24-benchmark-automsarima-point.npy` and the competitor rows from
`2026-09-21-benchmark-arima-point.npy`. Every table carries a `Source` column.

**Timing caveat.** This rerun was done on a different machine (4 cores, Python
3.11) from the saved runs (20 workers, Python 3.14), so times across sources are
not directly comparable.


In [1]:
import glob, os, subprocess, sys
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
LEVELS = np.round(np.arange(0.01, 1.0, 0.01), 2)

HERE = os.getcwd()
PAPER = os.path.join(HERE, "smooth-paper")

SMOOTH = ["AutoMSARIMA Back", "AutoMSARIMA Opt"]
COMPETITORS = ["statsforecast AutoARIMA", "skforecast Arima"]
EXCLUDED = ["pmdarima auto_arima", "aeon AutoARIMA"]

## Versions

In [2]:
from importlib.metadata import version as pkg_version

print(f"python {sys.version.split()[0]}")
for _p in ("smooth", "numpy", "pandas", "statsforecast", "skforecast",
           "pmdarima", "aeon", "sktime", "fcompdata"):
    try:
        print(f"{_p:15s} v{pkg_version(_p)}")
    except Exception:
        print(f"{_p:15s} not installed")

print("\ngit branch / HEAD:")
for cmd in (["git", "rev-parse", "--abbrev-ref", "HEAD"], ["git", "log", "-1", "--format=%h %s"]):
    print("  " + subprocess.run(cmd, capture_output=True, text=True).stdout.strip())

python 3.11.15
smooth          v1.0.9
numpy           v2.4.6
pandas          v3.0.6
statsforecast   not installed
skforecast      not installed
pmdarima        not installed
aeon            not installed
sktime          not installed
fcompdata       v0.1.4

git branch / HEAD:
  arima-initialisation
  f3fddbc Start the ARIMA AR / MA parameters from Hannan-Rissanen estimates


## Datasets

In [3]:
from fcompdata import M1, M3, Tourism

datasets = ([M1[k] for k in M1.keys()]
            + [M3[k] for k in M3.keys()]
            + [Tourism[k] for k in Tourism.keys()])
print(f"M1: {len(M1)}   M3: {len(M3)}   Tourism: {len(Tourism)}   total: {len(datasets)}")

M1: 1001   M3: 3003   Tourism: 1311   total: 5315


## Run or load

The smooth rows come from `smooth-paper/run_full_benchmark.py` restricted to the
`AutoMSARIMA` rows (`BENCH_KIND=arima BENCH_ONLY=automsarima`); set
`BENCH_RERUN_ARIMA=1` to recompute them here. The competitor rows are read from
the arrays of the last full run, which included every implementation.

In [4]:
if os.environ.get("BENCH_RERUN_ARIMA", "0") == "1":
    import multiprocessing
    workers = os.environ.get("BENCH_WORKERS", str(min(20, multiprocessing.cpu_count())))
    subprocess.run([sys.executable, "run_full_benchmark.py"], cwd=PAPER, check=True,
                   env=dict(os.environ, BENCH_ONLY="automsarima", BENCH_WORKERS=workers,
                            OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
                            MKL_NUM_THREADS="1"))


def find(pattern):
    # Files matching `pattern` in this directory or in smooth-paper/, oldest first.
    hits = glob.glob(os.path.join(HERE, pattern)) + glob.glob(os.path.join(PAPER, pattern))
    if not hits:
        raise FileNotFoundError(pattern)
    return sorted(hits, key=os.path.basename)


def load_point(point_file):
    # A run is a point array and the method names in array order.
    methods_file = point_file.replace("-point.npy", "-methods.txt")
    if not os.path.exists(methods_file):
        methods_file = find(os.path.basename(methods_file)[:10] + "*-methods.txt")[-1]
    with open(methods_file) as f:
        names = [line.rstrip("\n").split("\t")[0] for line in f if line.strip()]
    return {"names": names, "point": np.load(point_file),
            "label": os.path.basename(point_file)[:10], "file": os.path.basename(point_file)}


rerun = load_point(find("*-benchmark-automsarima-point.npy")[-1])      # today's run, HR initials
previous = load_point(find("2026-09-24-benchmark-automsarima-point.npy")[-1])
saved = load_point(find("2026-09-21-benchmark-arima-point.npy")[-1])
for tag, run in (("HR rerun", rerun), ("previous smooth", previous), ("competitors", saved)):
    print(f"{tag:16s} {run['file']}  {run['point'].shape}  {', '.join(run['names'])}")

SOURCE, POINT = {}, {}
for name in SMOOTH:
    POINT[name + " HR"] = rerun["point"][rerun["names"].index(name)]
    SOURCE[name + " HR"] = rerun["label"]
    POINT[name] = previous["point"][previous["names"].index(name)]
    SOURCE[name] = previous["label"]
for name in COMPETITORS:
    POINT[name] = saved["point"][saved["names"].index(name)]
    SOURCE[name] = saved["label"]

# Guard against scoring a stale rerun.
SRC = os.path.abspath(os.path.join(HERE, "..", "..", "src", "smooth"))
newest_source = max(os.path.getmtime(os.path.join(folder, f))
                    for folder, _, files in os.walk(SRC) for f in files if f.endswith(".py"))
stale = os.path.getmtime(find("*-benchmark-automsarima-point.npy")[-1]) < newest_source
print("\nsmooth arrays postdate the smooth sources:", not stale)
assert not stale, "rerun the smooth rows: BENCH_RERUN_ARIMA=1"


HR rerun         2026-09-27-benchmark-automsarima-point.npy  (2, 5315, 3)  AutoMSARIMA Back, AutoMSARIMA Opt
previous smooth  2026-09-24-benchmark-automsarima-point.npy  (2, 5315, 3)  AutoMSARIMA Back, AutoMSARIMA Opt
competitors      2026-09-21-benchmark-arima-point.npy  (6, 5315, 3)  AutoMSARIMA Back, AutoMSARIMA Opt, statsforecast AutoARIMA, pmdarima auto_arima, skforecast Arima, aeon AutoARIMA

smooth arrays postdate the smooth sources: True


## Point accuracy and computational time

In [5]:
def table(names, values, extra=None, label=""):
    # Five-number summary + mean of `values(name)` for each named method.
    frame = pd.DataFrame({
        "Method": names,
        "Source": [SOURCE[n] for n in names],
        "Min": [np.nanmin(values(n)) for n in names],
        "Q1": [np.nanquantile(values(n), 0.25) for n in names],
        "Med": [np.nanmedian(values(n)) for n in names],
        "Q3": [np.nanquantile(values(n), 0.75) for n in names],
        "Max": [np.nanmax(values(n)) for n in names],
        "Mean": [np.nanmean(values(n)) for n in names],
    })
    for column, fun in (extra or {}).items():
        frame[column] = [fun(n) for n in names]
    print("\n" + "=" * 118 + f"\n{label}\n" + "=" * 118)
    print(frame.sort_values("Mean").to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return frame


COMPARED = [n + " HR" for n in SMOOTH] + SMOOTH + COMPETITORS
n_series = POINT[COMPARED[0]].shape[0]

rmsse_summary = table(
    COMPARED, lambda n: POINT[n][:, 0],
    extra={"Mean Time (s)": lambda n: np.nanmean(POINT[n][:, 1]),
           "Failed": lambda n: int(np.sum(np.isnan(POINT[n][:, 0])))},
    label=f"RMSSE, all {n_series} series (lower = better)")


RMSSE, all 5315 series (lower = better)
                 Method     Source    Min     Q1    Med     Q3     Max   Mean  Mean Time (s)  Failed
statsforecast AutoARIMA 2026-09-21 0.0246 0.6826 1.1849 2.3533 51.6160 1.9809         1.3597       0
       skforecast Arima 2026-09-21 0.0246 0.6927 1.1914 2.3703 51.6160 1.9971         2.0991       0
    AutoMSARIMA Back HR 2026-09-27 0.0246 0.7011 1.2015 2.3783 51.6160 2.0319         0.1591       0
        AutoMSARIMA Opt 2026-09-24 0.0246 0.7186 1.2155 2.4329 51.6160 2.0323         1.1959       0
       AutoMSARIMA Back 2026-09-24 0.0246 0.7024 1.2021 2.3767 51.6160 2.0343         0.2833       0
     AutoMSARIMA Opt HR 2026-09-27 0.0246 0.7216 1.2154 2.4480 51.6160 2.0561         0.7082       0


In [6]:
time_summary = table(
    COMPARED, lambda n: POINT[n][:, 1],
    extra={"Total (min)": lambda n: np.nansum(POINT[n][:, 1]) / 60,
           "Timeouts": lambda n: int(np.nansum(POINT[n][:, 2]))},
    label=f"Computational time in seconds, all {n_series} series")


Computational time in seconds, all 5315 series
                 Method     Source    Min     Q1    Med     Q3      Max   Mean  Total (min)  Timeouts
    AutoMSARIMA Back HR 2026-09-27 0.0235 0.0689 0.1418 0.1993   1.1585 0.1591      14.0948         0
       AutoMSARIMA Back 2026-09-24 0.0292 0.1218 0.2323 0.3585   2.2418 0.2833      25.0979         0
     AutoMSARIMA Opt HR 2026-09-27 0.0407 0.1536 0.5688 1.0911   6.8619 0.7082      62.7362         0
        AutoMSARIMA Opt 2026-09-24 0.0557 0.2828 0.9365 1.7121   6.4421 1.1959     105.9353         0
statsforecast AutoARIMA 2026-09-21 0.0311 0.2736 0.5402 1.2872  70.1680 1.3597     120.4508         0
       skforecast Arima 2026-09-21 0.0277 0.3790 0.6861 1.8508 101.2417 2.0991     185.9452         0


## Change against the previous smooth run

The same 5,315 series, before (2026-09-24, previous initials) and after (Hannan-Rissanen
initials). Times are from different machines; see the caveat above.

In [7]:
rows = []
for name in SMOOTH:
    before, after = POINT[name], POINT[name + " HR"]
    d = after[:, 0] - before[:, 0]
    rows.append({
        "Method": name,
        "Med RMSSE before": np.nanmedian(before[:, 0]),
        "Med RMSSE after": np.nanmedian(after[:, 0]),
        "Mean RMSSE before": np.nanmean(before[:, 0]),
        "Mean RMSSE after": np.nanmean(after[:, 0]),
        "Series changed": int(np.sum(~np.isclose(before[:, 0], after[:, 0], equal_nan=True))),
        "Better": int(np.sum(d < -1e-8)),
        "Worse": int(np.sum(d > 1e-8)),
        "Mean time before": np.nanmean(before[:, 1]),
        "Mean time after": np.nanmean(after[:, 1]),
    })
print(f"\nbefore = {previous['label']}   after = {rerun['label']}")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))



before = 2026-09-24   after = 2026-09-27
          Method  Med RMSSE before  Med RMSSE after  Mean RMSSE before  Mean RMSSE after  Series changed  Better  Worse  Mean time before  Mean time after
AutoMSARIMA Back            1.2021           1.2015             2.0343            2.0319            2981    1751   1728            0.2833           0.1591
 AutoMSARIMA Opt            1.2155           1.2154             2.0323            2.0561            3226    1734   1818            1.1959           0.7082
